In [ ]:
## Population proportion Calculation

from google.colab import drive
import pandas as pd
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set paths
input_path = "/content/drive/MyDrive/SW_IFL/RevisedDeviationLevel.csv"
output_folder = "/content/drive/MyDrive/SW_IFL/Output/"

# 3. Define variables (EXCLUDING INCOME)
base_vars = ['TPOP', 'WH', 'BAA', 'HL', 'OTHER', 'OccHUnits', 'RENTER',
            'AGE60Nabv', 'POVERTY']  # Removed 'INCOME'
years = ['1980', '1990', '2000', '2010', '2020']
geo_cols = ['GISJOIN', 'GEOID10', 'GEOID']  # Key geographic identifiers

# 4. Load and process data
if not os.path.exists(input_path):
    print(f"Error: File not found at {input_path}")
else:
    df = pd.read_csv(input_path)

    # Verify geo columns exist
    missing_geo = [col for col in geo_cols if col not in df.columns]
    if missing_geo:
        print(f"Warning: Missing geographic columns {missing_geo}")

    # Calculate city totals (retaining geo columns)
    city_totals = df.groupby('ID').agg({
        **{f'{var}_{year}': 'sum' for var in base_vars for year in years},
        **{geo: 'first' for geo in geo_cols if geo in df.columns}  # Keep first instance
    }).reset_index()

    # Calculate proportions (retaining original geo columns)
    prop_df = df[['ID'] + [col for col in geo_cols if col in df.columns]].copy()

    for year in years:
        for var in base_vars:
            col_name = f'{var}_{year}'
            prop_col = f'{var}_prop_{year}'

            merged = df[['ID', col_name]].merge(
                city_totals[['ID', col_name]].rename(columns={col_name: 'city_total'}),
                on='ID'
            )
            prop_df[prop_col] = (merged[col_name] / merged['city_total']).round(6)

    # Save results
    os.makedirs(output_folder, exist_ok=True)
    city_totals.to_csv(output_folder + "City_Totals_Exposure.csv", index=False)
    prop_df.to_csv(output_folder + "Tract_Proportions_Exposure.csv", index=False)

    print(f"""
    PROCESSING COMPLETE!
    Output files (with geographic identifiers):
    1. City totals: {output_folder}City_Totals_Exposure.csv
    2. Tract proportions: {output_folder}Tract_Proportions_Exposure.csv

    Columns preserved in both files:
    - ID
    - {', '.join([col for col in geo_cols if col in df.columns])}
    """)

    # Show column verification
    print("\nProportions file columns preview:")
    print(prop_df.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

    PROCESSING COMPLETE!
    Output files (with geographic identifiers):
    1. City totals: /content/drive/MyDrive/SW_IFL/Output/City_Totals_Exposure.csv
    2. Tract proportions: /content/drive/MyDrive/SW_IFL/Output/Tract_Proportions_Exposure.csv
    
    Columns preserved in both files:
    - ID
    - GISJOIN, GEOID10, GEOID
    

Proportions file columns preview:
['ID', 'GISJOIN', 'GEOID10', 'GEOID', 'TPOP_prop_1980', 'WH_prop_1980', 'BAA_prop_1980', 'HL_prop_1980', 'OTHER_prop_1980', 'OccHUnits_prop_1980', 'RENTER_prop_1980', 'AGE60Nabv_prop_1980', 'POVERTY_prop_1980', 'TPOP_prop_1990', 'WH_prop_1990', 'BAA_prop_1990', 'HL_prop_1990', 'OTHER_prop_1990', 'OccHUnits_prop_1990', 'RENTER_prop_1990', 'AGE60Nabv_prop_1990', 'POVERTY_prop_1990', 'TPOP_prop_2000', 'WH_prop_2000', 'BAA_prop_2000', 'HL_prop_2000', 'OTHER_prop_2000', 'OccHUnits_prop_2000', 'RENTER

In [ ]:
## Summarize Population based on

from google.colab import drive
import pandas as pd
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set file paths
input_path = "/content/drive/MyDrive/SW_IFL/NationalExposureULTIMATE.csv"
output_folder = "/content/drive/MyDrive/SW_IFL/Complete_City_Summaries/"
os.makedirs(output_folder, exist_ok=True)

# 3. Load data with progress monitoring
print("Loading data...")
try:
    df = pd.read_csv(input_path)
    print(f"Successfully loaded {len(df)} rows")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

# 4. Verify critical columns exist
required_columns = ['ID'] + \
                  [f'{var}_prop_{year}' for var in ['WH', 'BAA', 'HL'] for year in ['1980', '1990']] + \
                  ['DT_Max_1980', 'DT_Min_1980', 'NDVI_1980', 'Elevation_M']  # Sample check
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    print(f"Error: Missing required columns: {missing_cols[:5]}...")
else:
    print("All required columns present")

# 5. Define analysis parameters
base_vars = ['WH', 'BAA', 'HL', 'OTHER', 'OccHUnits', 'RENTER', 'AGE60Nabv', 'POVERTY']
years = ['1980', '1990', '2000', '2010', '2020']

# 6. Process all cities with progress tracking
print(f"\nProcessing {df['ID'].nunique()} cities...")
processed = 0

for city_id, city_data in df.groupby('ID'):
    try:
        # Initialize results
        results = []

        # A. Process socio-demographic proportions (SUMS)
        for var in base_vars:
            row = {'Variable': f'% {var}', 'Type': 'Population Proportion (Sum)'}
            for year in years:
                prop_col = f'{var}_prop_{year}'
                for cond, mask in [
                    ('Max<0', city_data[f'DT_Max_{year}'] < 0),
                    ('Max>0', city_data[f'DT_Max_{year}'] > 0),
                    ('Min<0', city_data[f'DT_Min_{year}'] < 0),
                    ('Min>0', city_data[f'DT_Min_{year}'] > 0)
                ]:
                    row[f'{year}_{cond}'] = city_data.loc[mask, prop_col].sum()
            results.append(row)

        # B. Process NDVI (AVERAGES)
        ndvi_row = {
            'Variable': 'NDVI',
            'Type': 'Vegetation Index (Avg)',
            **{
                f'{year}_{cond}': city_data.loc[mask, f'NDVI_{year}'].mean()
                for year in years
                for cond, mask in [
                    ('Max<0', city_data[f'DT_Max_{year}'] < 0),
                    ('Max>0', city_data[f'DT_Max_{year}'] > 0),
                    ('Min<0', city_data[f'DT_Min_{year}'] < 0),
                    ('Min>0', city_data[f'DT_Min_{year}'] > 0)
                ]
            }
        }
        results.append(ndvi_row)

        # C. Process Elevation (CONDITION-SPECIFIC AVERAGES)
        elev_row = {
            'Variable': 'Elevation_M',
            'Type': 'Elevation (Avg)',
            **{
                f'{year}_{cond}': city_data.loc[mask, 'Elevation_M'].mean()
                for year in years
                for cond, mask in [
                    ('Max<0', city_data[f'DT_Max_{year}'] < 0),
                    ('Max>0', city_data[f'DT_Max_{year}'] > 0),
                    ('Min<0', city_data[f'DT_Min_{year}'] < 0),
                    ('Min>0', city_data[f'DT_Min_{year}'] > 0)
                ]
            }
        }
        results.append(elev_row)

        # Save city results
        safe_city_id = str(city_id).replace('/', '_')
        output_path = f"{output_folder}city_{safe_city_id}.csv"
        pd.DataFrame(results).to_csv(output_path, index=False)

        processed += 1
        if processed % 10 == 0:
            print(f"Processed {processed} cities...")

    except Exception as e:
        print(f"Error processing city {city_id}: {e}")

print(f"\nSuccessfully processed {processed} cities")
print(f"Results saved to: {output_folder}")

# 7. Verification
print("\nVerifying output...")
sample_files = [f for f in os.listdir(output_folder) if f.startswith('city_')][:3]
for f in sample_files:
    print(f"\nSample file: {f}")
    display(pd.read_csv(f"{output_folder}{f}").head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data...
Successfully loaded 33000 rows
All required columns present

Processing 39 cities...
Processed 10 cities...
Processed 20 cities...
Processed 30 cities...

Successfully processed 39 cities
Results saved to: /content/drive/MyDrive/SW_IFL/Complete_City_Summaries/

Verifying output...

Sample file: city_1.csv


,Variable,Type,1980_Max<0,1980_Max>0,1980_Min<0,1980_Min>0,1990_Max<0,1990_Max>0,1990_Min<0,1990_Min>0,...,2000_Min<0,2000_Min>0,2010_Max<0,2010_Max>0,2010_Min<0,2010_Min>0,2020_Max<0,2020_Max>0,2020_Min<0,2020_Min>0
0,% WH,Population Proportion (Sum),0.478842,0.521171,0.435294,0.564719,0.488623,0.511400,0.478035,0.521988,...,0.538000,0.461991,0.406875,0.593131,0.540013,0.459993,0.439158,0.560854,0.546196,0.453816
1,% BAA,Population Proportion (Sum),0.780900,0.219102,0.275565,0.724437,0.739036,0.260949,0.258897,0.741088,...,0.480366,0.519642,0.560367,0.439602,0.427551,0.572418,0.549231,0.450757,0.391233,0.608755
2,% HL,Population Proportion (Sum),0.457516,0.542479,0.319986,0.680009,0.476517,0.523497,0.377177,0.622837,...,0.415333,0.584666,0.408829,0.591188,0.421283,0.578734,0.443449,0.556537,0.429255,0.570731
3,% OTHER,Population Proportion (Sum),0.648636,0.351356,0.480201,0.519791,0.601087,0.398902,0.442058,0.557931,...,0.456095,0.543892,0.442672,0.557325,0.394466,0.605531,0.452504,0.547509,0.377009,0.623004
4,% OccHUnits,Population Proportion (Sum),0.553907,0.446088,0.425792,0.574203,0.551920,0.448077,0.456182,0.543815,...,0.522583,0.477414,0.450388,0.549622,0.495146,0.504864,0.465399,0.534595,0.466870,0.533124
5,% RENTER,Population Proportion (Sum),0.640448,0.359547,0.434975,0.565020,0.620411,0.379612,0.467647,0.532376,...,0.558493,0.441508,0.523398,0.476591,0.526787,0.473202,0.518510,0.481491,0.478744,0.521257
6,% AGE60Nabv,Population Proportion (Sum),0.618531,0.381477,0.484277,0.515731,0.594690,0.405305,0.505488,0.494507,...,0.549699,0.450297,0.449603,0.550413,0.516003,0.484013,0.458655,0.541303,0.485637,0.514321
7,% POVERTY,Population Proportion (Sum),0.628358,0.371647,0.409877,0.590128,0.608413,0.391586,0.411204,0.588795,...,0.515322,0.484683,0.488061,0.511917,0.478256,0.521722,0.499619,0.500366,0.471195,0.528790
8,NDVI,Vegetation Index (Avg),0.223327,0.245114,0.259990,0.218522,0.233447,0.239352,0.268768,0.212658,...,0.252494,0.228501,0.224211,0.269715,0.265928,0.232436,0.261284,0.305515,0.302743,0.266852
9,Elevation_M,Elevation (Avg),65.111958,70.979037,92.308562,53.016675,65.419761,71.034749,86.237841,54.930871,...,75.048505,60.797672,62.282176,73.046296,78.127275,58.495596,60.988883,74.751661,75.694033,61.040747



Sample file: city_2.csv


,Variable,Type,1980_Max<0,1980_Max>0,1980_Min<0,1980_Min>0,1990_Max<0,1990_Max>0,1990_Min<0,1990_Min>0,...,2000_Min<0,2000_Min>0,2010_Max<0,2010_Max>0,2010_Min<0,2010_Min>0,2020_Max<0,2020_Max>0,2020_Min<0,2020_Min>0
0,% WH,Population Proportion (Sum),0.136211,0.863794,0.135198,0.864807,0.165471,0.834543,0.157752,0.842262,...,0.073819,0.926188,0.129737,0.870255,0.066738,0.933254,0.174419,0.825572,0.099899,0.900092
1,% BAA,Population Proportion (Sum),0.008177,0.991842,0.020404,0.979615,0.009763,0.990230,0.049418,0.950575,...,0.005232,0.994760,0.012140,0.987863,0.006879,0.993124,0.018842,0.981155,0.020935,0.979062
2,% HL,Population Proportion (Sum),0.057937,0.942072,0.091930,0.908079,0.083551,0.916444,0.142108,0.857887,...,0.057308,0.942697,0.066672,0.933333,0.047346,0.952659,0.084208,0.915788,0.103163,0.896833
3,% OTHER,Population Proportion (Sum),0.047541,0.952459,0.094862,0.905138,0.043094,0.956909,0.085067,0.914936,...,0.021738,0.978258,0.031234,0.968764,0.034669,0.965329,0.051981,0.948013,0.058188,0.941806
4,% OccHUnits,Population Proportion (Sum),0.118175,0.881817,0.121534,0.878458,0.135808,0.864183,0.135310,0.864681,...,0.061774,0.938244,0.100825,0.899179,0.056764,0.943240,0.129569,0.870427,0.087421,0.912575
5,% RENTER,Population Proportion (Sum),0.103347,0.896657,0.133352,0.866652,0.099459,0.900536,0.096177,0.903818,...,0.051601,0.948401,0.083956,0.916049,0.058739,0.941266,0.089481,0.910506,0.101706,0.898281
6,% AGE60Nabv,Population Proportion (Sum),0.134161,0.865833,0.111091,0.888903,0.160440,0.839558,0.140196,0.859802,...,0.061899,0.938095,0.122709,0.877295,0.054883,0.945121,0.163628,0.836372,0.087678,0.912322
7,% POVERTY,Population Proportion (Sum),0.091663,0.908328,0.152293,0.847698,0.093149,0.906849,0.106442,0.893556,...,0.043065,0.956930,0.063093,0.936922,0.052275,0.947740,0.092295,0.907714,0.109063,0.890946
8,NDVI,Vegetation Index (Avg),0.404564,0.275737,0.406432,0.277294,0.380395,0.262405,0.333344,0.268243,...,0.471263,0.286564,0.449366,0.282258,0.448786,0.288987,0.469594,0.310739,0.438725,0.320135
9,Elevation_M,Elevation (Avg),1046.982939,34.731904,1032.714764,50.822540,944.023874,30.622445,673.995338,55.224106,...,1641.376412,58.520724,1167.866455,43.333024,1491.802680,63.119322,976.822472,31.769322,974.132974,64.996771



Sample file: city_3.csv


,Variable,Type,1980_Max<0,1980_Max>0,1980_Min<0,1980_Min>0,1990_Max<0,1990_Max>0,1990_Min<0,1990_Min>0,...,2000_Min<0,2000_Min>0,2010_Max<0,2010_Max>0,2010_Min<0,2010_Min>0,2020_Max<0,2020_Max>0,2020_Min<0,2020_Min>0
0,% WH,Population Proportion (Sum),0.594751,0.405255,0.498281,0.501725,0.524865,0.475121,0.556954,0.443032,...,0.577577,0.422415,0.516825,0.483069,0.615915,0.383979,0.511677,0.488324,0.662102,0.337899
1,% BAA,Population Proportion (Sum),0.805884,0.194208,0.181400,0.818692,0.702586,0.297436,0.255672,0.744350,...,0.317952,0.682050,0.577888,0.422104,0.354204,0.645788,0.544291,0.455696,0.715776,0.284211
2,% HL,Population Proportion (Sum),0.563106,0.436900,0.278993,0.721013,0.526111,0.473911,0.352431,0.647591,...,0.396552,0.603429,0.491643,0.508355,0.451919,0.548079,0.476475,0.523513,0.633255,0.366733
3,% OTHER,Population Proportion (Sum),0.640118,0.359826,0.288549,0.711395,0.553392,0.446597,0.382060,0.617929,...,0.413127,0.586856,0.530128,0.469865,0.480642,0.519351,0.523858,0.476121,0.627725,0.372254
4,% OccHUnits,Population Proportion (Sum),0.618400,0.381598,0.399108,0.600890,0.555562,0.444454,0.458471,0.541545,...,0.471157,0.528874,0.534050,0.465960,0.510134,0.489876,0.526805,0.473187,0.650606,0.349386
5,% RENTER,Population Proportion (Sum),0.674161,0.325830,0.309289,0.690702,0.612507,0.387475,0.370946,0.629036,...,0.385785,0.614200,0.607983,0.392031,0.431372,0.568642,0.597662,0.402329,0.659503,0.340488
6,% AGE60Nabv,Population Proportion (Sum),0.588332,0.411682,0.386793,0.613221,0.537401,0.462569,0.464189,0.535781,...,0.474908,0.525076,0.514312,0.485738,0.514833,0.485217,0.506965,0.493014,0.642093,0.357886
7,% POVERTY,Population Proportion (Sum),0.641456,0.358561,0.289354,0.710663,0.577014,0.422971,0.320410,0.679575,...,0.357701,0.642289,0.529765,0.470203,0.399965,0.600003,0.522323,0.477662,0.649995,0.349990
8,NDVI,Vegetation Index (Avg),0.174894,0.191281,0.205182,0.162143,0.092870,0.163681,0.183142,0.077026,...,0.225469,0.164339,0.168935,0.191074,0.205572,0.153937,0.216346,0.234853,0.226695,0.222278
9,Elevation_M,Elevation (Avg),105.726438,330.872968,332.548006,100.069167,99.768600,326.128459,326.268476,105.047714,...,321.394840,108.921489,86.241642,346.435642,293.203496,128.500751,91.833123,338.136496,249.368579,131.392713
